# Part 2.2 Tools

After this notebook you can give an agent tools: typed Python functions whose docstrings
are the instructions the model reads.

In [ ]:
from kea.config import MODEL, setup
from kea.notebook import ask_and_approve, final, steps

setup("kea-rebooking-local")

## See it

An agent is a model calling tools in a loop. This trace of the rebooking agent goes around
it: `get_booking`, `search_flights`, `rebook`, then the reply.

<img src="../../assets/diagrams/agent-loop.png" alt="The agent loop: request, model, tools, observation, result" width="560">

<img src="../../assets/screenshots/p2-02-tools-trace.png" alt="the LangSmith trace of a rebooking run: getbooking, searchflights, rebook, then the reply." width="900">

## Build it

The smallest agent: a model, three tools, and a one-line prompt. The tools are plain
functions in `kea.rebooking.tools`. We leave out `issue_credit` for now. It moves money,
and [2.8 Middleware](08_middleware.ipynb) puts it behind an approval.

In [ ]:
from deepagents import create_deep_agent

from kea.rebooking.tools import get_booking, rebook, search_flights

tools_agent = create_deep_agent(
    model=MODEL,
    tools=[get_booking, search_flights, rebook],
    system_prompt=(
        "You are Kea Air's rebooking assistant. Look up the booking, then rebook the "
        "passenger on the earliest flight after the cancelled one."
    ),
)

In [ ]:
result = await ask_and_approve(
    tools_agent,
    "My flight KA 1182 got cancelled. Confirmation ABC123, last name Example.",
)
steps(result)
print()
final(result)

## Under the hood

The model picks an action, the tool returns an observation, and the loop repeats until the
model answers instead of calling a tool.

The name, the docstring, and the type hints become the tool schema. The model sees it on
every call. This is what it reads for `search_flights`:

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(search_flights), indent=2))

The docstring is a prompt, so write it for the model. Hard rules don't belong in prompts.
A prompt persuades and code enforces. `rebook` refuses a full flight, a different route,
and a flight that leaves before the original, whatever the model asks for:

In [ ]:
import inspect

from kea.rebooking.tools import plan_rebooking

print(inspect.getsource(plan_rebooking))

- `get_booking` takes the last name as well as the code for the same reason: the check
  lives in the tool, not in the prompt.
- The core logic (`plan_rebooking`) is a pure function. The tool around it only reads the
  fixtures and the date. That keeps the rules testable without a model.
- Tools return errors as data (`{"error": ...}`). The model reads the error and recovers,
  for example by picking another flight.

Docs: [Deep Agents tools](https://docs.langchain.com/oss/python/deepagents/tools).

## Recap

- A tool is a typed function. Its docstring is the description the model reads.
- Put hard rules in the tool's code, not in the prompt.
- Return errors as data so the model can recover.

**Next:** [Part 2.3 MCP](03_mcp.ipynb)